# CIFAR-10 Image Classification Using CNN

Original notebook cleaned for GitHub. For a reproducible script-based run, use `python train.py`.

In [48]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision 
from torchvision.datasets import CIFAR10

In [56]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize =>(-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root= "./data", train= True, download = True, transform = transform)
testset = CIFAR10(root= "./data", train= False, download = True, transform = transform)

In [50]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64, shuffle = False)

### Build the CNN

In [51]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # Block 1: 3 -> 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 32 -> 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc_layers(x)
        return x


In [37]:
model = CNN()

In [52]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [39]:
# Training the CNN

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        # Forward propagation
        output = model(images)
        # Calculate loss
        loss = criterion(output, labels)
        # Backpropagation
        loss.backward()
        # Update model parameters
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 1.4024245924961842
epoch = 2/10 & loss = 0.9615584657244061
epoch = 3/10 & loss = 0.7711317480906196
epoch = 4/10 & loss = 0.6482479427095569
epoch = 5/10 & loss = 0.5423242287029086
epoch = 6/10 & loss = 0.4527508706578513
epoch = 7/10 & loss = 0.36820244593807805
epoch = 8/10 & loss = 0.29404662639055107
epoch = 9/10 & loss = 0.22774621855724803
epoch = 10/10 & loss = 0.17889418144284003


In [57]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 41.79
